In [ ]:
# Paths (relative to the repository root, so the notebooks run on any machine)
# Expected layout:  <repo>/notebooks/ (this file), <repo>/data/, <repo>/models/,
#                   <repo>/figures/, <repo>/results/, <repo>/feature_importance/
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
FI_DIR = ROOT / "feature_importance"
for _d in (DATA_DIR, MODELS_DIR, FIG_DIR, RESULTS_DIR, FI_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [1]:
# --- Cell 0: imports (unchanged) ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from scipy.stats import pearsonr, spearmanr

In [2]:
# --- Cell 1: load the pre-split, leakage-free datasets from corrected 05_ ---
train_df = pd.read_csv(str(DATA_DIR / "Final_ML_Train.csv"))
test_df  = pd.read_csv(str(DATA_DIR / "Final_ML_Test.csv"))

train_df = train_df.dropna(subset=["pIC50"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["pIC50"]).reset_index(drop=True)
print(train_df.shape, test_df.shape)

(21440, 694) (5361, 694)


In [3]:
print(train_df.index[:5])

RangeIndex(start=0, stop=5, step=1)


In [4]:
# --- Cell 2: separate features/target, same exclude list as before ---
target = "pIC50"
exclude = ["SMILES_Ligands", "Counterion", "Cell_line", "Metal", "IC50_Dark_value", "pIC50"]

X_train_raw = train_df.drop(columns=exclude)
X_test_raw  = test_df.drop(columns=exclude)
y_train = train_df[target]
y_test  = test_df[target]

print(X_train_raw.shape, X_test_raw.shape)   # expect (21440, 688) (5361, 688)

(21440, 688) (5361, 688)


In [14]:
y_train.describe()

count    21440.000000
mean         4.722610
std          0.804107
min          2.088470
25%          4.032440
50%          4.640165
75%          5.251812
max          8.698970
Name: pIC50, dtype: float64

In [15]:
y_test.describe()

count    5361.000000
mean        4.708522
std         0.784066
min         2.580540
25%         4.041102
50%         4.623423
75%         5.221126
max         7.522879
Name: pIC50, dtype: float64

In [5]:
# --- Cell 3: Time(h) missing-value fill — TRAIN-ONLY median ---
time_median = X_train_raw["Time(h)"].median()
X_train_raw["Time(h)"] = X_train_raw["Time(h)"].fillna(time_median)
X_test_raw["Time(h)"]  = X_test_raw["Time(h)"].fillna(time_median)   # frozen train value

print(X_train_raw.isnull().sum().sum(), X_test_raw.isnull().sum().sum())

0 0


In [6]:
# --- Cell 4: remove Ipc (numerically unstable descriptor) ---
X_train_raw = X_train_raw.drop(columns=["Ipc"])
X_test_raw  = X_test_raw.drop(columns=["Ipc"])
print(X_train_raw.shape, X_test_raw.shape)   # expect (21440, 687) (5361, 687)

(21440, 687) (5361, 687)


In [7]:
# --- Cell 5: confirm no leakage columns remain (should now be empty) ---
leakage_cols = [c for c in X_train_raw.columns if "IC50" in c]
print("Leakage columns found:", leakage_cols)   # expect: []

Leakage columns found: []


In [8]:
# --- Cell 6: drop Year and Time(h) (not intended as modeling features) ---
X_train_raw = X_train_raw.drop(columns=["Year", "Time(h)"])
X_test_raw  = X_test_raw.drop(columns=["Year", "Time(h)"])
print(X_train_raw.shape, X_test_raw.shape)   # expect (21440, 685) (5361, 685)

(21440, 685) (5361, 685)


In [9]:
# --- Cell 7: one-hot encode Metal/Cell_line — fit categories on TRAIN, align test ---
cat_train = pd.get_dummies(train_df[["Metal", "Cell_line"]], drop_first=True)
cat_test  = pd.get_dummies(test_df[["Metal", "Cell_line"]], drop_first=True)

# align test dummy columns to train's (any train category missing in test -> 0 column;
# any test category not seen in train is dropped, since the model never learned it)
cat_test = cat_test.reindex(columns=cat_train.columns, fill_value=0)
print(cat_train.shape, cat_test.shape)

(21440, 695) (5361, 695)


In [12]:
# --- Cell 7 (revised): fit one-hot encoding on the FULL known category vocabulary ---
all_categories = pd.concat([train_df[["Metal", "Cell_line"]], test_df[["Metal", "Cell_line"]]])
cat_all = pd.get_dummies(all_categories, drop_first=True)

cat_train = cat_all.loc[train_df.index]
cat_test  = cat_all.loc[test_df.index]

print(cat_train.shape, cat_test.shape)   # should now both show the same 757 columns

(26801, 757) (10722, 757)


In [18]:
# --- Cell 7 (revised): fit one-hot encoding on the FULL known category vocabulary ---
all_categories = pd.concat(
    [train_df[["Metal", "Cell_line"]], test_df[["Metal", "Cell_line"]]],
    ignore_index=True   # avoids duplicate 0..N indices colliding
)
cat_all = pd.get_dummies(all_categories, drop_first=True)

n_train = len(train_df)
cat_train = cat_all.iloc[:n_train].reset_index(drop=True)
cat_test  = cat_all.iloc[n_train:].reset_index(drop=True)

print(cat_train.shape, cat_test.shape)   # both should now show the same 757 columns

(21440, 757) (5361, 757)


In [19]:
# --- Cell 8: assemble final feature matrices ---
X_train = pd.concat([X_train_raw.reset_index(drop=True), cat_train.reset_index(drop=True)], axis=1)
X_test  = pd.concat([X_test_raw.reset_index(drop=True),  cat_test.reset_index(drop=True)],  axis=1)

print(X_train.shape, X_test.shape)   # expect (21440, 1442) (5361, 1442)

(21440, 1442) (5361, 1442)


In [22]:
print("Unique cell lines in train_df:", train_df["Cell_line"].nunique())
print("Unique cell lines in test_df:", test_df["Cell_line"].nunique())
print("Unique cell lines overall:", pd.concat([train_df["Cell_line"], test_df["Cell_line"]]).nunique())

print("Unique metals in train_df:", train_df["Metal"].nunique())
print("Unique metals in test_df:", test_df["Metal"].nunique())

print(cat_train.shape, cat_test.shape)   # before the reindex step

Unique cell lines in train_df: 692
Unique cell lines in test_df: 404
Unique cell lines overall: 754
Unique metals in train_df: 5
Unique metals in test_df: 5
(21440, 757) (5361, 757)


In [23]:
# --- Cell 9: shared CV splitter (fixes B4/§2.4 consistency issue at the same time) ---
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)

In [24]:
# --- Cell 10: hyperparameter search ---
param_dist = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "bootstrap": [True]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=cv_splitter,        # was cv=5 (unshuffled default)
    scoring="r2",
    random_state=42,
    verbose=2,
    n_jobs=-2
)
random_search.fit(X_train, y_train)

print(random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
{'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 30, 'bootstrap': True}
Best CV R²: 0.7485433618027069


In [15]:
# --- Cell 11: evaluate best model ---
best_rf = random_search.best_estimator_
train_pred = best_rf.predict(X_train)
test_pred  = best_rf.predict(X_test)

def evaluate(y_true, y_pred, dataset):
    print(f"\n{dataset}")
    print("R²   :", round(r2_score(y_true, y_pred), 4))
    print("MAE  :", round(mean_absolute_error(y_true, y_pred), 4))
    print("RMSE :", round(root_mean_squared_error(y_true, y_pred), 4))

evaluate(y_train, train_pred, "Training")
evaluate(y_test, test_pred, "Testing")

In [27]:
# --- Cell 12: save everything downstream notebooks need ---
joblib.dump(best_rf, str(MODELS_DIR / "Best_RandomForest_Model1.pkl"))

X_train.to_csv(str(DATA_DIR / "X_train1.csv"), index=False)
X_test.to_csv(str(DATA_DIR / "X_test1.csv"), index=False)
y_train.to_csv(str(DATA_DIR / "y_train1.csv"), index=False)
y_test.to_csv(str(DATA_DIR / "y_test1.csv"), index=False)

with open(str(DATA_DIR / "RF_Best_Parameters1.txt"), "w") as f:
    f.write(str(random_search.best_params_))

In [3]:
# load the saved best model

best_model = joblib.load(str(MODELS_DIR / "Best_RandomForest_Model1.pkl"))

In [5]:
# load the Train/Test Dataset

X_train = pd.read_csv(str(DATA_DIR / "X_train1.csv"))
X_test = pd.read_csv(str(DATA_DIR / "X_test1.csv"))

y_train = pd.read_csv(str(DATA_DIR / "y_train1.csv")).squeeze()
y_test = pd.read_csv(str(DATA_DIR / "y_test1.csv")).squeeze()

In [6]:
# Generate the prediction

y_train_pred = best_model.predict(X_train)

y_test_pred = best_model.predict(X_test)

In [7]:
# Calculate R²
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

In [8]:
# Calculate RMSE
train_rmse = root_mean_squared_error(y_train, y_train_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)

In [9]:
# Calculate MAE
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

In [10]:
# Calculate Pearson correlation

train_pearson, train_pearson_p = pearsonr(
    y_train,
    y_train_pred
)

test_pearson, test_pearson_p = pearsonr(
    y_test,
    y_test_pred
)

In [11]:
# Calculate Spearman correlation

train_spearman, train_spearman_p = spearmanr(
    y_train,
    y_train_pred
)

test_spearman, test_spearman_p = spearmanr(
    y_test,
    y_test_pred
)

In [12]:
# Create the manuscript table

results = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "R²": [train_r2, test_r2],
    "RMSE": [train_rmse, test_rmse],
    "MAE": [train_mae, test_mae],
    "Pearson r": [train_pearson, test_pearson],
    "Pearson p-value": [train_pearson_p, test_pearson_p],
    "Spearman ρ": [train_spearman, test_spearman],
    "Spearman p-value": [train_spearman_p, test_spearman_p]
})

results = results.round(4)

results

,Dataset,R²,RMSE,MAE,Pearson r,Pearson p-value,Spearman ρ,Spearman p-value
0,Training,0.9426,0.1926,0.1340,0.9727,0.0,0.9725,0.0
1,Testing,0.7748,0.3721,0.2586,0.8805,0.0,0.8765,0.0


In [13]:
# Save the table

results.to_csv(
    str(RESULTS_DIR / "Randomforest_Performance_Table1.csv"),
    index=False
)

In [14]:
with open(
    str(DATA_DIR / "RF_Results1.txt"),
    "w"
) as f:
    f.write(f"Training R2 = {train_r2:.4f}\n")
    f.write(f"Testing R2 = {test_r2:.4f}\n")
    f.write(f"Testing MAE = {test_mae:.4f}\n")
    f.write(f"Testing RMSE = {test_rmse:.4f}\n")
    f.write(f"Best CV R2 = {random_search.best_score_:.4f}\n")